In [1]:
import pandas as pd
import os
import plotnine as p9
from plotnine import *
from scipy.stats import spearmanr

# ==============================
# 输入文件
# ==============================
bed_files = [
    "/Users/zhanghaojiang/Downloads/_用所选项目新建的文件夹54/coded/CUT.Tag/GSE217860/HT29/HT29.bed",
    "/Users/zhanghaojiang/Downloads/_用所选项目新建的文件夹54/coded/CHIP-seq/GSE99205/HaCaT_BG4_rep1/HaCaT.bed",

]

all_data = []
ann_list = []

# 第一步：循环读取并计算各自的统计量
for input_bed in bed_files:
    Tname = os.path.basename(input_bed).replace(".bed", "")
    
    df = pd.read_csv(input_bed, sep=r"\s+", header=None)
    temp_df = pd.DataFrame()
    temp_df["signal"] = pd.to_numeric(df.iloc[:, 3], errors="coerce")
    temp_df["pqs_count"] = pd.to_numeric(df.iloc[:, 4], errors="coerce")
    temp_df = temp_df.dropna()
    temp_df = temp_df[temp_df["pqs_count"] > 0]
    
    # 计算相关性
    r, pv = spearmanr(temp_df["pqs_count"], temp_df["signal"])
    
    # P 值格式化逻辑
    if pv < 1e-300:
        p_text = "P < 1e-300"
    elif pv < 0.001:
        p_text = f"P = {pv:.2e}"
    else:
        p_text = f"P = {pv:.3f}"
    
    # 动态计算标注位置
    ann_list.append({
        'source': Tname,
        'label': f"Spearman r = {r:.3f}\n{p_text}",
        'x': temp_df["pqs_count"].max() * 0.6,
        'y': temp_df["signal"].max() * 0.9
    })
    
    temp_df["source"] = Tname
    all_data.append(temp_df)

# 合并数据
combined_data = pd.concat(all_data)
ann_df = pd.DataFrame(ann_list)
combined_data["source"] = pd.Categorical(
    combined_data["source"],
    categories=["HaCaT", "HT29"],
    ordered=True
)

ann_df["source"] = pd.Categorical(
    ann_df["source"],
    categories=["HaCaT", "HT29"],
    ordered=True
)
# ==============================
# 绘图：布局优化
# ==============================
# 增加宽度到 18，让中间的散点图区域有更多空间伸展
p9.options.figure_size = (14, 6) 

plot_combined = (
    ggplot(combined_data, aes(x="pqs_count", y="signal"))
    + geom_bin2d(bins=60)
    + scale_fill_cmap(name="Count (log10)", trans="log10")
    + geom_smooth(method="lm", color="blue")
    
    # 关键：scales="free_y" 让 HaCaT 和 HT29 独立计算 Y 轴
    + facet_wrap("~source", nrow=1, scales="free_y")
    
    # 添加格式化后的相关性标注 (size=22 在大画布下通常比较协调)
    + geom_text(data=ann_df, mapping=aes(x='x', y='y', label='label'), 
                size=22, inherit_aes=False, fontweight="bold")
    
    + labs(
        x="Number of PQs",
        y="BG4 signal"
    )
    + theme_bw()
    + theme(
        text=element_text(family="Arial"),
        # 1. 坐标轴标题
        axis_title_x=element_text(size=24),
        axis_title_y=element_text(size=24),
        # 2. 坐标轴刻度数字
        axis_text_x=element_text(size=24, color="black"),
        axis_text_y=element_text(size=24, color="black"),
        # 3. 图例设置
        legend_text=element_text(size=20),
        legend_title=element_text(size=20),
        legend_position="right",
        legend_key_width=15, # 稍微收窄图例宽度，把空间留给主图
        # 4. 标题与背景
        strip_background=element_blank(), 
        strip_text=element_text(size=22, weight="bold"),
        # 5. 子图间距：设置在 0.4 左右能保证中间的 Y 轴数值不拥挤
        panel_spacing=0 
    )
)

print(plot_combined)
plot_combined.save("combined_plots_final.pdf", bbox_inches="tight")

<ggplot: (1400 x 600)>


/Users/zhanghaojiang/anaconda3/envs/mlx/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 14 x 6 in image.
/Users/zhanghaojiang/anaconda3/envs/mlx/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: combined_plots_final.pdf


In [ ]:
# Fig2 plotHeatmap (293T)

! computeMatrix scale-regions -S  /home/hjzhang/G4quard/todo/drawheatmap/ENCFF716SFD.bw  -R /home/hjzhang/G4quard/todo/cell200lsort/293T_hg38.bed --regionBodyLength 1000 -b 1000 -a 1000  --binSize 10 -o deeptools/293T_hg38_DNase.gz  -p 1200
! plotHeatmap --colorList 'white,red,#830101' -m deeptools/293T_hg38_DNase.gz  -o deeptools/293T_hg38_DNase.pdf --colorList 'white,red,#830101'


